In [ ]:
########################## CHOROPLETH WORLD MAP AND PERCENTAGE IPV4 #################
import pandas as pd
import pycountry
import plotly.express as px

# Load dataset
dataset = pd.read_csv("ip_alloc(in).csv")

# Group by country and sum IPv4 allocations
ip_pr_country = dataset.groupby("country name")["ipv4"].sum().reset_index()

# Extract the total IPv4 allocation from the 'World' row
world_total_row = ip_pr_country[ip_pr_country['country name'] == 'World']
if not world_total_row.empty:
    world_total = world_total_row['ipv4'].values[0]
else:
    world_total = ip_pr_country['ipv4'].sum()

# Calculate the percentage for each country
ip_pr_country['ipv4_pct'] = (ip_pr_country['ipv4'] / world_total) * 100

# Exclude 'World' in country name column
ip_pr_country = ip_pr_country[ip_pr_country["country name"] != "World"]

# Add ISO-3 codes to your DataFrame
def get_iso3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except LookupError:
        return None

ip_pr_country['iso_alpha'] = ip_pr_country['country name'].apply(get_iso3)
ip_pr_country.dropna(subset=['iso_alpha'], inplace=True)

# Define the custom green-yellow-orange-red color scale
custom_colorscale = [
    [0.0, 'rgb(0, 255, 0)'],       # Green
    [0.33, 'rgb(255, 255, 0)'],    # Yellow
    [0.66, 'rgb(255, 165, 0)'],    # Orange
    [1.0, 'rgb(255, 0, 0)']        # Red
]

# Create a Plotly choropleth map with the custom color scale
fig = px.choropleth(
    ip_pr_country,
    locations="iso_alpha",           # ISO 3-letter country codes
    color="ipv4_pct",                # Data to be color-scaled
    hover_name="country name",       # Displayed on hover
    color_continuous_scale=custom_colorscale,
    labels={'ipv4_pct': 'IPv4 % Share'},
    title="Global IPv4 Allocation as Percentage of World Total",
    projection="natural earth"
)

# Set the map's background to dark
fig.update_layout(
    template='plotly_dark',
    geo=dict(
        bgcolor='rgb(10,10,10)',
        showland=True,
        landcolor='rgb(10,10,10)',
        showocean=True,
        oceancolor='rgb(10,10,10)',
        showlakes=True,
        lakecolor='rgb(10,10,10)',
        showrivers=True,
        rivercolor='rgb(10,10,10)',
        projection_type='natural earth'
    ),
    margin={"r":0,"t":30,"l":0,"b":0}
)

fig.show()

fig.write_html("ipv4_allocation_choropleth_world_map.html")
